In [ ]:
!pip install pandas
!pip install numpy
!pip install duckdb
!pip install holidays

In [ ]:
import os
import duckdb
import holidays
import pandas as pd
import time
import requests

Fonte = https://dados.ons.org.br/dataset/geracao-usina-2


Perguntas de Pesquisa: 
 - É possível prever a geração de energia hidrelétrica utilizando dados históricos de geração?
 - Quais períodos do ano apresentam maior e menor geração de energia hidrelétrica?
 - O nível dos reservatórios está relacionado ao aumento do valor cobrado pela energia?

Hipoteses: 
 - O histórico de geração permite prever a geração futura
 -A geração possui padrões sazonais
 -Existe uma relação positiva entre o volume de precipitação e a geração hidrelétrica
 -Em epocas de estiagem, o valor da energia aumenta por causa da escassez



In [ ]:
import pandas as pd

df = pd.read_csv('data.csv', sep=';')
df.head()

In [ ]:
df.describe()

Variavel-alvo: val_geracao
Tipo de geração: Hidráulica
Regiao: Sul e Centro-Oeste
Granularidade: Horária
Período 2001 - 2020
Horizonte de previsão: Mensal

Por que escolher a Granularidade Mensal: 
    O conjunto de dados do ONS ja possui os registros, o que permite aproveitar a resolucaos. A geracao hidreletrica pode variar ao longo do tempo, e a granularidade mensal permitiria analisar e identificar isso de uma forma mais detalhada. Isso nos permitiria, alem de tudo, avaliar o impacto financeiro ao longo do tempo.

Horizonte de planejamento para o mes seguinte:
    Permite estimar a geracao do mes seguinte
    Possibilita utilizar os padroes e horarios para estimar e identificar tendencias. 
    Parece um horizonte adequado para um modelo inicial, sem tornar o problema muito complexo.

Nao foi possivel fazer a agregacao dos dados com o pandas porque devido ao grande volume de dados, o processamento apresentou limitacoes de memoria. Diante disso, optou-se pelo uso do DuckDB.
Foi realizado o processo de juncao dos 19 anos em um unico dataframe.

In [ ]:
import os
import duckdb

os.makedirs("dados_consolidados", exist_ok=True)
con = duckdb.connect()

In [ ]:
for ano_inicio in range(2001, 2021, 2):

    ano_fim = ano_inicio + 1

    arquivos = [
        f"dados_mensais/GERACAO_USINA-2_{ano_inicio}.parquet",
        f"dados_mensais/GERACAO_USINA-2_{ano_fim}.parquet"
    ]

    destino = f"dados_consolidados/geracao_{ano_inicio}_{ano_fim}.parquet"
    con.execute(f"""
        COPY (
        SELECT *
        FROM read_parquet({arquivos})
        )
        TO '{destino}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

In [ ]:
import duckdb
con.execute("""
    COPY (
    SELECT *
    FROM read_parquet('dados_consolidados/*.parquet')
    )
    TO 'geracao2001_2020.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

In [ ]:
import duckdb
con = duckdb.connect()

df = con.execute("""
SELECT *
FROM read_parquet('geracao2001_2020.parquet')
ORDER BY din_instante DESC
LIMIT 10
""").df()

df

Como vamos fazer uma analise Mensal, a seguir, se cria uma coluna com o ano e uma com o mes. 

In [ ]:
con.execute("""
    COPY (
    SELECT *,
    EXTRACT(YEAR FROM din_instante) AS Ano
    FROM read_parquet('geracao2001_2020.parquet')
    )
    TO 'geracao_modificada.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [ ]:
con.execute("""
    COPY(
    SELECT *,
    EXTRACT (MONTH FROM din_instante) AS MES
    FROM read_parquet('geracao_modificada.parquet')
    )
    TO 'geracao_temp.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [ ]:
import os
os.replace(
    'geracao_temp.parquet',
    'geracao_modificada.parquet'
)

Agora o arquivo modificado contem o ano e o mes.

Cumprindo com o segundo requisito, o arquivo precisa de datas de feriado. 

A biblioteca holidays contem feriados baseados na região, no nosso caso, Brasil 2001-2021.

Com isso, podemos projetar as datas no parquet. Por enquanto, utilizaremos apenas um booleano sobre se é ou não um feriado.

In [ ]:
feriados= holidays.Brazil(years= range(2001, 2021))
df_feriados= pd.DataFrame({'data_feriado': pd.to_datetime(list(feriados.keys()))})

In [ ]:
con= duckdb.connect()
con.execute("""
    COPY(
        SELECT temp.*, (CAST(temp.din_instante AS DATE) 
            IN (
                SELECT CAST(data_feriado AS DATE) 
                    FROM df_feriados)) 
                    AS feriado
            FROM read_parquet('geracao_modificada.parquet') temp
    )
    TO 'geracao_temp.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

os.replace('geracao_temp.parquet', 'geracao_modificada.parquet')

Cumprindo com o terceiro requisito, o arquivo precisa de dados climaticos referente as coordenadas aproximadas perto de grandes usinas. 

O Open-Meteo fornece dados climáticos diários a partir de coordenadas (latitude e longitude), com base na reanálise ERA5.


In [ ]:
pontos = {
    'PR': [(-25.41, -54.59), (-26.00, -51.70)],  
    'SC': [(-27.27, -52.38), (-27.60, -51.32)],   
    'RS': [(-28.26, -52.41), (-29.45, -53.28)],   
    'GO': [(-13.83, -48.30), (-18.41, -49.10)],   
    'MT': [(-14.87, -55.79), (-11.86, -55.50)],   
    'MS': [(-20.78, -51.63), (-20.44, -54.65)],   
    'DF': [(-15.79, -47.88)], 
}

In [ ]:
URL = "https://archive-api.open-meteo.com/v1/archive"

partes = []

for estado, coords in pontos.items():
    for lat, lon in coords:
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": "2001-01-01",
            "end_date": "2020-12-31",
            "daily": "precipitation_sum,temperature_2m_mean",
            "timezone": "America/Sao_Paulo",
        }
        r = requests.get(URL, params=params)
        r.raise_for_status()

        d= pd.DataFrame(r.json()['daily'])
        d["id_estado"] = estado
        partes.append(d)
        time.sleep(15)
df_clima = pd.concat(partes, ignore_index=True)
df_clima["time"] = pd.to_datetime(df_clima["time"])
df_clima.to_parquet("clima_diario.parquet")


In [ ]:
con.execute("""
    COPY (
        WITH por_estado_dia AS (
            SELECT id_estado, time AS data,
                   AVG(precipitation_sum)   AS chuva_mm,
                   AVG(temperature_2m_mean) AS temp_c
            FROM read_parquet('clima_diario.parquet')
            GROUP BY id_estado, time
        )
        SELECT id_estado,
               EXTRACT(YEAR FROM data)  AS Ano,
               EXTRACT(MONTH FROM data) AS MES,
               SUM(chuva_mm) AS chuva_mm_mes,
               AVG(temp_c)   AS temp_media_mes
        FROM por_estado_dia
        GROUP BY ALL
    )
    TO 'clima_mensal.parquet' (FORMAT PARQUET)
""")


In [ ]:
con.execute("""
    COPY (
        SELECT id_estado, Ano, MES,
               SUM(TRY_CAST(val_geracao AS DOUBLE)) AS geracao_total,
               COUNT(DISTINCT CASE WHEN feriado
                     THEN CAST(din_instante AS DATE) END) AS n_feriados
        FROM read_parquet('geracao_modificada.parquet')
        WHERE nom_tipocombustivel = 'Hidráulica'
          AND id_estado IN ('PR','SC','RS','GO','MT','MS','DF')
        GROUP BY id_estado, Ano, MES
    )
    TO 'geracao_mensal.parquet' (FORMAT PARQUET)
""")


In [ ]:
df_final = con.execute("""
    SELECT g.*, c.chuva_mm_mes, c.temp_media_mes
    FROM read_parquet('geracao_mensal.parquet') g
    LEFT JOIN read_parquet('clima_mensal.parquet') c
        USING (id_estado, Ano, MES)
    ORDER BY id_estado, Ano, MES
""").df()

print(len(df_final))           
print(df_final.isna().sum())     


Conforme solicitado pelo professor, o arquivo também recebe as bandeiras tarifárias, obtidas nos dados abertos da ANEEL.

A bandeira é definida mês a mês e vale para o país inteiro, por isso é unida aos dados apenas por ano e mês. Como o sistema de bandeiras só começou em janeiro de 2015, os meses de 2001 a 2014 ficam sem valor.

In [ ]:
URL_BANDEIRAS = "https://dadosabertos.aneel.gov.br/dataset/7f43a020-6dc5-44b8-80b4-d97eaa94436c/resource/0591b8f6-fe54-437b-b72b-1aa2efd46e42/download/bandeira-tarifaria-acionamento.csv"

df_bandeiras = pd.read_csv(URL_BANDEIRAS, sep=";", decimal=",")
df_bandeiras["NomBandeiraAcionada"].value_counts()


In [ ]:
df_bandeiras["DatCompetencia"] = pd.to_datetime(df_bandeiras["DatCompetencia"])
df_bandeiras["Ano"] = df_bandeiras["DatCompetencia"].dt.year
df_bandeiras["MES"] = df_bandeiras["DatCompetencia"].dt.month

df_bandeiras = df_bandeiras.rename(columns={
    "NomBandeiraAcionada": "bandeira",
    "VlrAdicionalBandeira": "adicional_bandeira",
})
df_bandeiras = df_bandeiras[["Ano", "MES", "bandeira", "adicional_bandeira"]]
df_bandeiras.to_parquet("bandeiras_mensal.parquet")


In [ ]:
df_final = con.execute("""
    SELECT g.*, c.chuva_mm_mes, c.temp_media_mes,
           b.bandeira, b.adicional_bandeira
    FROM read_parquet('geracao_mensal.parquet') g
    LEFT JOIN read_parquet('clima_mensal.parquet') c
        USING (id_estado, Ano, MES)
    LEFT JOIN read_parquet('bandeiras_mensal.parquet') b
        USING (Ano, MES)
    ORDER BY id_estado, Ano, MES
""").df()

print(len(df_final))
print(df_final.isna().sum())

Seguindo a recomendação de incluir indicadores econômicos, utilizamos o consumo de energia elétrica do Brasil, total e industrial, em GWh, obtido no Sistema Gerenciador de Séries Temporais (SGS) do Banco Central.

Enquanto o clima representa a água disponível para gerar energia, o consumo representa a demanda. Como a rede elétrica é interligada, os valores são nacionais e também são unidos por ano e mês.


In [ ]:
def baixar_sgs(codigo, nome):
    url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
           "?formato=json&dataInicial=01/01/2001&dataFinal=31/12/2020")
    d = pd.read_json(url)
    d["data"] = pd.to_datetime(d["data"], format="%d/%m/%Y")
    d["Ano"] = d["data"].dt.year
    d["MES"] = d["data"].dt.month
    return d.rename(columns={"valor": nome})[["Ano", "MES", nome]]


In [ ]:
df_consumo = baixar_sgs(1406, "consumo_total_gwh")
df_industrial = baixar_sgs(1404, "consumo_industrial_gwh")

df_economia = df_consumo.merge(df_industrial, on=["Ano", "MES"])
df_economia.to_parquet("economia_mensal.parquet")
df_economia.head()


In [ ]:
df_final = con.execute("""
    SELECT g.*, c.chuva_mm_mes, c.temp_media_mes,
           b.bandeira, b.adicional_bandeira,
           e.consumo_total_gwh, e.consumo_industrial_gwh
    FROM read_parquet('geracao_mensal.parquet') g
    LEFT JOIN read_parquet('clima_mensal.parquet') c
        USING (id_estado, Ano, MES)
    LEFT JOIN read_parquet('bandeiras_mensal.parquet') b
        USING (Ano, MES)
    LEFT JOIN read_parquet('economia_mensal.parquet') e
        USING (Ano, MES)
    ORDER BY id_estado, Ano, MES
""").df()

print(len(df_final))
print(df_final.isna().sum())
df_final.to_parquet("df_final.parquet", index=False)

1. A seguir sera observada a Hipotese de se a geracao possui padroes sazonais

In [ ]:
import duckdb
con = duckdb.connect()

df = con.execute("""
SELECT *
FROM read_parquet('df_final.parquet')
LIMIT 10
""").df()

df

In [ ]:
import duckdb
con = duckdb.connect()
df = con.execute("""
COPY(
    SELECT *,
    CASE   
        WHEN MES IN (12, 1, 2) THEN 'VERAO'
        WHEN MES IN (3, 4, 5) THEN 'OUTONO'
        WHEN MES IN (6, 7, 8) THEN 'INVERNO'
        WHEN MES IN (9, 10, 11) THEN 'PRIMAVERA'
    END AS ESTACAO
    FROM read_parquet('df_final.parquet')
    )
    TO 'df_final_estacoes.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""").df()

df


Cria uma coluna nova com a estacao do mes em questao para testar a hipotese de padroes sazonais de geracao

In [ ]:
import duckdb
con = duckdb.connect()
df = con.execute("""
SELECT * 
FROM read_parquet('df_final_estacoes.parquet')
LIMIT 10
""").df()
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

con = duckdb.connect()
df = con.execute("""
SELECT AVG(geracao_total) as MEDIA_GERACAO, ESTACAO, ANO
FROM read_parquet('df_final_estacoes.parquet')
GROUP BY ANO, ESTACAO
ORDER BY MEDIA_GERACAO DESC
""").df()
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

con = duckdb.connect()

df = con.execute("""
    SELECT
        ESTACAO,
        geracao_total
    FROM read_parquet('df_final_estacoes.parquet')
    WHERE geracao_total IS NOT NULL
""").df()

ordem = ['VERAO', 'OUTONO', 'INVERNO', 'PRIMAVERA']

plt.figure(figsize=(11, 6))

sns.boxplot(
    data=df,
    x='ESTACAO',
    y='geracao_total',
    order=ordem
)

plt.title('Distribuição da Geração de Energia por Estação')
plt.xlabel('Estação do Ano')
plt.ylabel('Geração de Energia')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_media = con.execute("""
    SELECT
        ESTACAO,
        AVG(geracao_total) AS MEDIA_GERACAO
    FROM read_parquet('df_final_estacoes.parquet')
    GROUP BY ESTACAO
    ORDER BY MEDIA_GERACAO DESC
""").df()

plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_media,
    x='ESTACAO',
    y='MEDIA_GERACAO',
    order=df_media['ESTACAO']
)

plt.title('Geração Média de Energia por Estação')
plt.xlabel('Estação do Ano')
plt.ylabel('Geração Média')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

E possivel observar que a primavera concentra os maiores valores de geracao. 

2. A Seguir sera observada a hipotese de se existe uma relação positiva entre o volume de precipitação e a geração hidrelétrica

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_correlacao = con.execute("""
    SELECT chuva_mm_mes, geracao_total, Ano
    FROM read_parquet('df_final_estacoes.parquet')
    WHERE chuva_mm_mes IS NOT NULL AND
        geracao_total IS NOT NULL
""").df()

x = df_correlacao['chuva_mm_mes']
y = df_correlacao['geracao_total']

plt.figure(figsize=(10,6))
plt.scatter(x, y, alpha=0.5)
plt.title('Geracao X Precipitacao')
plt.xlabel('Geracao de Energia')
plt.ylabel('Precipitacao')
plt.grid(alpha=0.3)
plt.show()


O Scatter plot fica muito complicado de visualizar, por isso optei por plotar um heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df_final_estacoes = pd.read_parquet('df_final_estacoes.parquet')
correlacoes = df_final_estacoes.groupby('id_estado').apply(
    lambda grupo: grupo['chuva_mm_mes'].corr(
        grupo['geracao_total']
    )
).reset_index(name='CORRELACAO')

matriz = correlacoes.set_index('id_estado')

plt.figure(figsize=(8, 10))

sns.heatmap(
    matriz,
    annot=True,
    cmap='RdYlGn',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    fmt='.2f'
)

plt.title('Correlação entre Chuvas e Geração por Estado')
plt.xlabel('')
plt.ylabel('Estado')
plt.tight_layout()
plt.show()

3. A seguir sera comparada a geracao hidreletrica diaria em feriados versus dias fora de feriado

In [ ]:
import duckdb

con = duckdb.connect()

df_feriado_resumo = con.execute("""
    WITH diario AS (
        SELECT
            CAST(din_instante AS DATE) AS data,
            feriado,
            SUM(TRY_CAST(val_geracao AS DOUBLE)) AS geracao_dia
        FROM read_parquet('geracao_modificada.parquet')
        WHERE nom_tipocombustivel = 'Hidráulica'
          AND id_estado IN ('PR','SC','RS','GO','MT','MS','DF')
        GROUP BY data, feriado
    )
    SELECT
        CASE WHEN feriado THEN 'Feriado' ELSE 'Dia Comum' END AS TIPO_DIA,
        COUNT(*) AS QTD_DIAS,
        ROUND(AVG(geracao_dia), 2) AS MEDIA_GERACAO_DIA,
        ROUND(MEDIAN(geracao_dia), 2) AS MEDIANA_GERACAO_DIA
    FROM diario
    GROUP BY feriado
    ORDER BY MEDIA_GERACAO_DIA DESC
""").df()

df_feriado_resumo

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_feriado_diario = con.execute("""
    SELECT
        CAST(din_instante AS DATE) AS data,
        CASE WHEN feriado THEN 'Feriado' ELSE 'Dia Comum' END AS TIPO_DIA,
        SUM(TRY_CAST(val_geracao AS DOUBLE)) AS GERACAO_DIA
    FROM read_parquet('geracao_modificada.parquet')
    WHERE nom_tipocombustivel = 'Hidráulica'
      AND id_estado IN ('PR','SC','RS','GO','MT','MS','DF')
    GROUP BY data, feriado
""").df()

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_feriado_diario,
    x='TIPO_DIA',
    y='GERACAO_DIA',
    order=['Dia Comum', 'Feriado']
)

plt.title('Distribuição da Geração Diária: Dia Comum vs. Feriado')
plt.xlabel('Tipo de Dia')
plt.ylabel('Geração Diária')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_feriado_resumo,
    x='TIPO_DIA',
    y='MEDIA_GERACAO_DIA',
    order=['Dia Comum', 'Feriado']
)

plt.title('Geração Média Diária: Dia Comum vs. Feriado')
plt.xlabel('Tipo de Dia')
plt.ylabel('Geração Média Diária')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Observa-se que nos dias de feriado a geracao hidreletrica diaria media cai cerca de 19,1% em relacao aos dias comuns (de 352.380 para 285.108), refletindo a reducao da demanda eletrica comercial e industrial nessas datas.

Para provar que dias onde a industria e comercio estão fechados levam a uma geração menor, vamos comparar a geração em dias da semana vs fim de semana

In [ ]:
import duckdb

con = duckdb.connect()

df_semana_resumo = con.execute("""
    WITH diario AS (
        SELECT
            CAST(din_instante AS DATE) AS data,
            EXTRACT(ISODOW FROM CAST(din_instante AS DATE)) AS dia_semana,
            SUM(TRY_CAST(val_geracao AS DOUBLE)) AS geracao_dia
        FROM read_parquet('geracao_modificada.parquet')
        WHERE nom_tipocombustivel = 'Hidráulica'
          AND id_estado IN ('PR','SC','RS','GO','MT','MS','DF')
        GROUP BY data
    )
    SELECT
        CASE WHEN dia_semana IN (6, 7) THEN 'Fim de Semana' ELSE 'Dia da Semana' END AS PERIODO_SEMANA,
        COUNT(*) AS QTD_DIAS,
        ROUND(AVG(geracao_dia), 2) AS MEDIA_GERACAO_DIA,
        ROUND(MEDIAN(geracao_dia), 2) AS MEDIANA_GERACAO_DIA
    FROM diario
    GROUP BY PERIODO_SEMANA
    ORDER BY MEDIA_GERACAO_DIA DESC
""").df()

df_semana_resumo

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_semana_diario = con.execute("""
    SELECT
        CAST(din_instante AS DATE) AS data,
        CASE
            WHEN EXTRACT(ISODOW FROM CAST(din_instante AS DATE)) IN (6, 7) THEN 'Fim de Semana'
            ELSE 'Dia da Semana'
        END AS PERIODO_SEMANA,
        SUM(TRY_CAST(val_geracao AS DOUBLE)) AS GERACAO_DIA
    FROM read_parquet('geracao_modificada.parquet')
    WHERE nom_tipocombustivel = 'Hidráulica'
      AND id_estado IN ('PR','SC','RS','GO','MT','MS','DF')
    GROUP BY data
""").df()

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_semana_diario,
    x='PERIODO_SEMANA',
    y='GERACAO_DIA',
    order=['Dia da Semana', 'Fim de Semana']
)

plt.title('Distribuição da Geração Diária: Dia da Semana vs. Fim de Semana')
plt.xlabel('Período da Semana')
plt.ylabel('Geração Diária')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_semana_resumo,
    x='PERIODO_SEMANA',
    y='MEDIA_GERACAO_DIA',
    order=['Dia da Semana', 'Fim de Semana']
)

plt.title('Geração Média Diária: Dia da Semana vs. Fim de Semana')
plt.xlabel('Período da Semana')
plt.ylabel('Geração Média Diária')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Observa-se que nos finais de semana a geracao hidreletrica diaria media e menor em comparacao aos dias da semana, acompanhando a reducao da atividade industrial e comercial aos sabados e domingos.